In [1]:
import importlib.util

required_packages = ("numpy", "pandas", "networkx", "scipy", "psutil")
missing_packages = [name for name in required_packages if importlib.util.find_spec(name) is None]
if missing_packages:
    raise ImportError("Missing packages. Run: %pip install " + " ".join(missing_packages))


In [2]:
from pathlib import Path

PROJECT = Path(r"C:\Users\LX\Desktop\quebec")
OUTPUT_ROOT = PROJECT / "L_revision"

# Pin the corrected input run used in the manuscript.
INPUT_MANIFEST = (PROJECT / "revision/fixed_10m_routing_release/07_exact_routing_inputs"
                  / "runs/20260926T235556_632053Z/manifest.json")
MODE = "full"
PILOT_OD_COUNT = 20
MAIN_DETOUR_PERCENT = 20
PROGRESS_EVERY = 100
LIMITS = {
    "seconds": 300,
    "labels": 3_000_000,
    "predecessor_links": 12_000_000,
    "rss_gib": 8,
    "available_gib": 2,
}

# None creates a new run. To resume, use the run directory printed at startup.
RESUME_RUN = None


In [3]:
from pathlib import Path
from datetime import datetime, timezone
from bisect import bisect_left, bisect_right
from dataclasses import dataclass
import gc
import gzip
import hashlib
import heapq
import json
import math
import os
import platform
import sys
import time
import traceback

import numpy as np
import pandas as pd
import networkx as nx
import psutil
import scipy
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.sparse import coo_matrix

SOLVER_VERSION = "exact_labels_fixed10m_v3"
BUDGET_PERCENTAGES = (5, 10, 15, 20)


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    opener = gzip.open if path.name.endswith(".gz") else open
    with opener(temporary, "wt", encoding="utf-8") as stream:
        json.dump(value, stream, ensure_ascii=False, separators=(",", ":"), allow_nan=False)
    temporary.replace(path)


def load_json(path):
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt", encoding="utf-8") as stream:
        return json.load(stream)


def atomic_csv(path, rows, columns=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    pd.DataFrame(rows, columns=columns, dtype=object).to_csv(
        temporary, index=False, encoding="utf-8-sig", lineterminator="\n")
    for attempt in range(3):
        try:
            temporary.replace(path)
            return
        except PermissionError as error:
            if attempt == 2:
                raise PermissionError(f"Cannot replace {path}. Close applications using this file and resume the run.") from error
            time.sleep(0.5)


@dataclass
class RoutingNetwork:
    n: int
    u: list
    v: list
    distance: list
    physical: list
    arc_ids: list
    out_ptr: list
    out_order: list
    in_ptr: list
    in_order: list
    distance_places: int = 0

    def outgoing(self, node):
        for position in range(self.out_ptr[node], self.out_ptr[node + 1]):
            yield self.out_order[position]

    def incoming(self, node):
        for position in range(self.in_ptr[node], self.in_ptr[node + 1]):
            yield self.in_order[position]


class ResourceLimit(RuntimeError):
    pass


class Monitor:
    """RSS is sampled, so the reported maximum is a sampled process RSS peak."""
    def __init__(self, limits):
        self.limits = limits
        self.started = time.perf_counter()
        self.last_sample = -math.inf
        self.process = psutil.Process()
        self.peak_rss = 0
        self.phase = "initializing"
        self.tick(force=True)

    def tick(self, force=False):
        elapsed = time.perf_counter() - self.started
        if elapsed > self.limits["seconds"]:
            raise ResourceLimit(f"Task time limit: {self.limits['seconds']} s")
        if force or elapsed - self.last_sample >= 0.5:
            self.last_sample = elapsed
            rss = self.process.memory_info().rss
            self.peak_rss = max(self.peak_rss, rss)
            if rss > self.limits["rss_gib"] * 1024**3:
                raise ResourceLimit(f"Process RSS limit: {self.limits['rss_gib']} GiB")
            if psutil.virtual_memory().available < self.limits["available_gib"] * 1024**3:
                raise ResourceLimit(f"Available memory below {self.limits['available_gib']} GiB")

    def statistics(self):
        return dict(seconds=time.perf_counter() - self.started,
                    sampled_peak_process_rss_gib=self.peak_rss / 1024**3,
                    last_phase=self.phase)


def reverse_lex_dijkstra(net, target, first, second, monitor=None):
    """Exact Python integers. Each tie chooses the smallest forward arc ID."""
    cost = [None] * net.n
    next_arc = [-1] * net.n
    cost[target] = (0, 0)
    queue = [(0, 0, target)]
    popped = 0
    while queue:
        a, b, node = heapq.heappop(queue)
        if cost[node] != (a, b):
            continue
        popped += 1
        if monitor and popped % 1024 == 0:
            monitor.tick()
        for edge in net.incoming(node):
            origin = net.u[edge]
            candidate = (a + first[edge], b + second[edge])
            if cost[origin] is None or candidate < cost[origin]:
                cost[origin] = candidate
                next_arc[origin] = edge
                heapq.heappush(queue, (*candidate, origin))
            elif candidate == cost[origin] and (next_arc[origin] < 0 or edge < next_arc[origin]):
                next_arc[origin] = edge
    return cost, next_arc


def restore_baseline(net, source, target, next_arc):
    route, seen = [], set()
    node = source
    while node != target:
        require(node not in seen, "Cycle in baseline route")
        seen.add(node)
        edge = next_arc[node]
        require(edge >= 0 and net.u[edge] == node, "Discontinuous baseline route or unreachable OD pair")
        route.append(edge)
        node = net.v[edge]
    return route


def check_route(net, exposure, route, source, target):
    node, d, c = source, 0, 0
    seen = {source}
    for edge in route:
        require(net.u[edge] == node, "Discontinuous route")
        node = net.v[edge]
        require(node not in seen, "Cycle in efficient route")
        seen.add(node)
        d += net.distance[edge]
        c += exposure[edge]
    require(node == target, "Incorrect route destination")
    return d, c


In [4]:
def compact_dag(label_node, label_d, label_c, predecessors, terminals, monitor):
    needed, stack = set(terminals), list(terminals)
    visited = 0
    while stack:
        label = stack.pop()
        visited += 1
        if visited % 2048 == 0:
            monitor.tick()
        for previous, edge in predecessors[label]:
            if previous not in needed:
                needed.add(previous)
                stack.append(previous)
    # Creation order is NOT necessarily topological: equal-cost links can arrive later.
    order = sorted(needed, key=lambda x: (label_d[x], label_c[x], x))
    lookup = {old: new for new, old in enumerate(order)}
    require(order[0] == 0, "Missing DAG root")
    dag = dict(node=[], distance_integer=[], exposure_integer=[], predecessors=[],
               terminals=[lookup[x] for x in terminals], root=0)
    for new, old in enumerate(order):
        if new % 2048 == 0:
            monitor.tick()
        links = sorted((lookup[p], edge) for p, edge in predecessors[old])
        require(all(p < new for p, _ in links), "Invalid DAG topological order")
        dag["node"].append(label_node[old])
        dag["distance_integer"].append(str(label_d[old]))
        dag["exposure_integer"].append(str(label_c[old]))
        dag["predecessors"].append(links)
    return dag


def representative_route(dag, terminal, monitor=None):
    """Lexicographically smallest complete forward arc-ID sequence for this terminal."""
    successors = {}
    seen, stack = {terminal}, [terminal]
    while stack:
        label = stack.pop()
        if monitor and len(seen) % 2048 == 0:
            monitor.tick()
        for previous, edge in dag["predecessors"][label]:
            successors.setdefault(previous, []).append((edge, label))
            if previous not in seen:
                seen.add(previous)
                stack.append(previous)
    node, route = dag["root"], []
    while node != terminal:
        edge, node = min(successors[node])
        route.append(edge)
    return route


def physical_intersection(net, dag, reference_route, monitor):
    # The intersection must be a subset of any one represented route.
    candidates = sorted({net.physical[e] for e in reference_route})
    bits = {physical: 1 << i for i, physical in enumerate(candidates)}
    all_bits = (1 << len(candidates)) - 1
    masks = [0] * len(dag["node"])
    counts = [0] * len(masks)
    counts[dag["root"]] = 1
    for label, links in enumerate(dag["predecessors"]):
        if label % 2048 == 0:
            monitor.tick()
        if label == dag["root"]:
            continue
        intersection, count = all_bits, 0
        for previous, edge in links:
            intersection &= masks[previous] | bits.get(net.physical[edge], 0)
            count = min(2, count + counts[previous])
        masks[label], counts[label] = intersection, count
    intersection, count = all_bits, 0
    for terminal in dag["terminals"]:
        intersection &= masks[terminal]
        count = min(2, count + counts[terminal])
    common = [physical for physical in candidates if intersection & bits[physical]]
    return common, count


In [5]:
def exact_pareto(net, exposure, source, target, limits):
    """All nondominated routes under 120% distance, represented by a predecessor DAG.

    Strict dominance deletes a state; identical cost states merge predecessor links.
    Positive arc distance permits distance-first label-setting and eliminates cycles.
    A resource limit returns no frontier or PEC result.
    """
    stats = dict(labels_created=0, predecessor_links_created=0, labels_expanded=0,
                 heap_peak=0, equal_cost_merges=0, bound_prunes=0, dominated_prunes=0)
    monitor = None
    try:
        monitor = Monitor(limits)
        monitor.phase = "reverse_shortest_baselines"
        short_cost, short_next = reverse_lex_dijkstra(net, target, net.distance, exposure, monitor)
        min_cost, min_next = reverse_lex_dijkstra(net, target, exposure, net.distance, monitor)
        require(short_cost[source] is not None and min_cost[source] is not None, "Unreachable OD pair")
        d_short, c_short = short_cost[source]
        c_min, d_min = min_cost[source]
        budget = 120 * d_short // 100
        shortest = restore_baseline(net, source, target, short_next)
        minimum = restore_baseline(net, source, target, min_next)
        require(check_route(net, exposure, shortest, source, target) == (d_short, c_short), "Shortest-route cost mismatch")
        require(check_route(net, exposure, minimum, source, target) == (d_min, c_min), "Minimum-exposure route cost mismatch")
        seeds = [(d_short, c_short)]
        if d_min <= budget:
            seeds.append((d_min, c_min))

        monitor.phase = "exact_labels"
        label_node, label_d, label_c = [source], [0], [0]
        predecessors = [[]]
        alive, processed = bytearray([1]), bytearray([0])
        front_d, front_ids = [[] for _ in range(net.n)], [[] for _ in range(net.n)]
        front_d[source], front_ids[source] = [0], [0]
        queue = [(0, 0, 0)]
        stats["labels_created"] = 1

        def lower_bound_pruned(node, d, c):
            if short_cost[node] is None:
                return True
            lower_d = d + short_cost[node][0]
            if lower_d > budget:
                return True
            lower_c = c + min_cost[node][0]
            for sd, sc in seeds:
                if sd <= lower_d and sc <= lower_c and (sd < lower_d or sc < lower_c):
                    return True
            pos = bisect_right(front_d[target], lower_d) - 1
            if pos >= 0:
                existing = front_ids[target][pos]
                return label_c[existing] <= lower_c and (
                    label_d[existing] < lower_d or label_c[existing] < lower_c)
            return False

        def append_link(label, previous, edge):
            if stats["predecessor_links_created"] >= limits["predecessor_links"]:
                raise ResourceLimit(f"Predecessor-link limit: {limits['predecessor_links']:,}")
            predecessors[label].append((previous, edge))
            stats["predecessor_links_created"] += 1

        def insert(node, d, c, previous, edge):
            distances, ids = front_d[node], front_ids[node]
            pos = bisect_left(distances, d)
            if pos < len(ids) and distances[pos] == d:
                existing = ids[pos]
                if label_c[existing] == c:
                    require(not processed[existing], "Equal-cost predecessor arrived after label expansion")
                    append_link(existing, previous, edge)
                    stats["equal_cost_merges"] += 1
                    return
                if label_c[existing] < c:
                    stats["dominated_prunes"] += 1
                    return
            if pos and label_c[ids[pos - 1]] <= c:
                stats["dominated_prunes"] += 1
                return
            end = pos
            while end < len(ids) and label_c[ids[end]] >= c:
                old = ids[end]
                require(not processed[old], "Expanded label dominated by a later label")
                alive[old] = 0
                predecessors[old].clear()
                end += 1
            if len(label_node) >= limits["labels"]:
                raise ResourceLimit(f"Created-label limit: {limits['labels']:,}")
            label = len(label_node)
            distances[pos:end], ids[pos:end] = [d], [label]
            label_node.append(node)
            label_d.append(d)
            label_c.append(c)
            predecessors.append([])
            alive.append(1)
            processed.append(0)
            append_link(label, previous, edge)
            heapq.heappush(queue, (d, c, label))
            stats["labels_created"] = len(label_node)
            stats["heap_peak"] = max(stats["heap_peak"], len(queue))

        pops = 0
        while queue:
            d, c, label = heapq.heappop(queue)
            pops += 1
            if pops % 1024 == 0:
                monitor.tick()
            if not alive[label]:
                continue
            node = label_node[label]
            if lower_bound_pruned(node, d, c):
                stats["bound_prunes"] += 1
                continue
            processed[label] = 1
            if node == target:
                continue
            stats["labels_expanded"] += 1
            for edge in net.outgoing(node):
                destination = net.v[edge]
                nd, nc = d + net.distance[edge], c + exposure[edge]
                if lower_bound_pruned(destination, nd, nc):
                    stats["bound_prunes"] += 1
                    continue
                insert(destination, nd, nc, label, edge)
        terminals = front_ids[target]
        require(terminals, "Search returned no feasible destination label")
        monitor.phase = "dag_and_physical_intersection"
        dag = compact_dag(label_node, label_d, label_c, predecessors, terminals, monitor)
        # Free the complete search state before constructing output and intersections.
        frontier = [(label_d[x], label_c[x]) for x in terminals]
        label_node.clear(); label_d.clear(); label_c.clear(); predecessors.clear()
        front_d.clear(); front_ids.clear()
        representative = representative_route(dag, dag["terminals"][0], monitor)
        require(representative == shortest, "Shortest-route tie-breaking differs from DAG reconstruction")
        common, route_count = physical_intersection(net, dag, representative, monitor)
        selected = {}
        for pct in BUDGET_PERCENTAGES:
            index = bisect_right([d for d, c in frontier], (100 + pct) * d_short // 100) - 1
            terminal = dag["terminals"][index]
            route = representative_route(dag, terminal, monitor)
            require(check_route(net, exposure, route, source, target) == frontier[index], "Budget-route cost mismatch")
            selected[str(pct)] = dict(terminal=terminal, arc_indices=route,
                distance_integer=str(frontier[index][0]), exposure_integer=str(frontier[index][1]))
        monitor.tick(force=True)
        stats.update(monitor.statistics(), frontier_points=len(frontier), dag_labels=len(dag["node"]),
                     dag_links=sum(map(len, dag["predecessors"])))
        return dict(status="completed", complete_frontier_within_20pct=True,
                    frontier_scope="all nondominated routes with D <= 1.20 * shortest distance",
                    shortest=dict(arc_indices=shortest, distance_integer=str(d_short), exposure_integer=str(c_short)),
                    minimum_exposure_unbounded=dict(arc_indices=minimum, distance_integer=str(d_min), exposure_integer=str(c_min)),
                    budget_20_distance_integer=str(budget), dag=dag,
                    selected_budget_routes=selected, common_physical_segments=common,
                    route_count_capped_at_two=route_count, multiple_routes=route_count > 1,
                    pec_physical_segments=common if route_count > 1 else [], stats=stats)
    except ResourceLimit as error:
        if monitor:
            stats.update(monitor.statistics())
        else:
            stats.update(seconds=0.0, sampled_peak_process_rss_gib=0.0, last_phase="initializing")
        return dict(status="incomplete_resource_limit", complete_frontier_within_20pct=False,
                    reason=str(error), stats=stats)


In [6]:
def tiny_network(n, edges, distance_places=0):
    """edges: (u, v, positive distance, nonnegative exposure, physical segment)."""
    u, v, d, c, physical = map(list, zip(*edges))
    out_order = sorted(range(len(u)), key=lambda e: (u[e], e))
    in_order = sorted(range(len(u)), key=lambda e: (v[e], e))
    out_ptr = [0] + np.cumsum(np.bincount(u, minlength=n)).tolist()
    in_ptr = [0] + np.cumsum(np.bincount(v, minlength=n)).tolist()
    return RoutingNetwork(n, u, v, d, physical,
        [f"edge_{i:06d}" for i in range(len(u))], out_ptr, out_order, in_ptr, in_order, distance_places), c


def enumerate_simple_routes(net, exposure, source, target):
    """Independent enumeration, only for small correctness fixtures."""
    routes, stack = [], [(source, frozenset([source]), (), 0, 0)]
    while stack:
        node, seen, route, d, c = stack.pop()
        if node == target:
            routes.append((d, c, route))
            continue
        for edge in net.outgoing(node):
            destination = net.v[edge]
            if destination not in seen:
                stack.append((destination, seen | {destination}, route + (edge,),
                              d + net.distance[edge], c + exposure[edge]))
    return routes


def enumerate_tiny_dag(dag):
    paths = [[()]] + [[] for _ in dag["node"][1:]]
    for label, links in enumerate(dag["predecessors"]):
        for previous, edge in links:
            paths[label].extend(p + (edge,) for p in paths[previous])
    return {route for label in dag["terminals"] for route in paths[label]}


def milp_budget_value(net, exposure, source, target, budget, seconds=10):
    """Independent float MILP cross-check on small integer fixtures, not exact certification."""
    m = len(net.u)
    rows = net.u + net.v + [net.n] * m
    cols = list(range(m)) * 3
    values = [1.0] * m + [-1.0] * m + [float(d) for d in net.distance]
    matrix = coo_matrix((values, (rows, cols)), shape=(net.n + 1, m)).tocsc()
    demand = np.zeros(net.n)
    demand[source], demand[target] = 1, -1
    lower, upper = np.r_[demand, -np.inf], np.r_[demand, float(budget)]
    result = milp(np.asarray(exposure, dtype=float), integrality=np.ones(m),
                  bounds=Bounds(0, 1), constraints=LinearConstraint(matrix, lower, upper),
                  options={"time_limit": float(seconds), "mip_rel_gap": 0.0})
    require(result.success, f"MILP fixture not solved to optimality: {result.message}")
    used = np.flatnonzero(result.x > 0.5).tolist()
    require(sum(net.distance[e] for e in used) <= budget, "MILP route exceeds integer distance budget")
    return sum(exposure[e] for e in used)


def run_algorithm_validation():
    limits = dict(seconds=60, labels=100000, predecessor_links=300000,
                  rss_gib=32, available_gib=0)
    fixtures = [
        ("unsupported_point_and_parallel_ties", 2,
         [(0, 1, d, c, f"p{i}") for i, (d, c) in enumerate(
             [(100, 100), (105, 95), (110, 60), (120, 50), (110, 60), (123, 0)])]),
        ("physical_intersection_across_different_branch_positions", 5,
         [(0, 1, 5, 5, "X"), (1, 3, 5, 5, "Y"), (0, 2, 5, 5, "Z"),
          (2, 3, 5, 5, "X"), (3, 4, 100, 100, "C")]),
        ("zero_exposure_positive_distance_cycle", 4,
         [(0, 1, 5, 0, "A"), (1, 2, 1, 0, "B"), (2, 1, 1, 0, "B"),
          (1, 3, 95, 0, "C"), (0, 3, 100, 1, "D")]),
        ("integers_beyond_int64_and_exact_boundary", 2,
         [(0, 1, 10**25, 10**27, "A"), (0, 1, 12 * 10**24, 0, "B"),
          (0, 1, 12 * 10**24 + 1, 0, "C")]),
    ]
    rng = np.random.default_rng(20260924)
    for index in range(24):
        n = 6
        edges = [(i, i + 1, int(rng.integers(3, 15)), int(rng.integers(0, 20)), f"p{i}")
                 for i in range(n - 1)]
        for edge_index in range(14):
            u, v = map(int, rng.choice(n, size=2, replace=False))
            edges.append((u, v, int(rng.integers(1, 25)), int(rng.integers(0, 30)),
                          f"p{int(rng.integers(0, 12))}"))
        fixtures.append((f"random_multigraph_{index:02d}", n, edges))
    checks = []
    for name, n, edges in fixtures:
        net, costs = tiny_network(n, edges)
        routes = enumerate_simple_routes(net, costs, 0, n - 1)
        shortest = min(routes)
        feasible = [p for p in routes if 100 * p[0] <= 120 * shortest[0]]
        efficient = [p for p in feasible if not any(q[0] <= p[0] and q[1] <= p[1]
                      and (q[0] < p[0] or q[1] < p[1]) for q in feasible)]
        expected_points = sorted({(d, c) for d, c, p in efficient})
        expected_routes = {p for d, c, p in efficient}
        result = exact_pareto(net, costs, 0, n - 1, limits)
        require(result["status"] == "completed", f"{name}: Validation run incomplete")
        dag = result["dag"]
        points = [(int(dag["distance_integer"][t]), int(dag["exposure_integer"][t])) for t in dag["terminals"]]
        require(points == expected_points, f"{name}: Pareto objective points differ")
        require(enumerate_tiny_dag(dag) == expected_routes, f"{name}: Missing equal-cost routes or parallel arcs")
        common = set.intersection(*[{net.physical[e] for e in route} for route in expected_routes])
        require(set(result["common_physical_segments"]) == common, f"{name}: Physical-segment intersection mismatch")
        require(result["route_count_capped_at_two"] == min(2, len(expected_routes)), f"{name}: Route-multiplicity classification mismatch")
        min_exposure = min(routes, key=lambda p: (p[1], p[0], p[2]))
        require(tuple(result["shortest"]["arc_indices"]) == shortest[2], f"{name}: Shortest-route tie-breaking mismatch")
        require(tuple(result["minimum_exposure_unbounded"]["arc_indices"]) == min_exposure[2], f"{name}: Unbounded minimum-exposure baseline mismatch")
        milp_checks = 0
        for pct in BUDGET_PERCENTAGES:
            best = min((p for p in routes if 100 * p[0] <= (100 + pct) * shortest[0]),
                       key=lambda p: (p[1], p[0], p[2]))
            require(tuple(result["selected_budget_routes"][str(pct)]["arc_indices"]) == best[2],
                    f"{name}: {pct}%budget-route mismatch")
            if name != "integers_beyond_int64_and_exact_boundary":
                value = milp_budget_value(net, costs, 0, n - 1, (100 + pct) * shortest[0] // 100)
                require(value == best[1], f"{name}: {pct}%budget MILP objective mismatch")
                milp_checks += 1
        checks.append(dict(fixture=name, passed=True, enumerated_routes=len(routes),
                           efficient_routes=len(expected_routes), cost_points=len(points), milp_checks=milp_checks))
    net, costs = tiny_network(2, fixtures[0][2])
    for cap in (dict(labels=1), dict(predecessor_links=0), dict(seconds=-1)):
        result = exact_pareto(net, costs, 0, 1, {**limits, **cap})
        require(result["status"] == "incomplete_resource_limit" and "dag" not in result
                and "pec_physical_segments" not in result, "Resource-limited task returned a complete frontier")
    checks.append(dict(fixture="resource_limits_never_emit_complete_frontier", passed=True,
                       enumerated_routes=0, efficient_routes=0, cost_points=0, milp_checks=0))
    return dict(status="passed", fixture_count=len(checks),
                milp_budget_checks=sum(r["milp_checks"] for r in checks), checks=checks)


In [7]:
def load07(project, input_manifest=None):
    pointer = None
    if input_manifest is None:
        pointer_path = project / "revision/fixed_10m_routing_release/07_exact_routing_inputs/LATEST_SUCCESS.json"
        require(pointer_path.is_file(), f"Input success pointer not found: {pointer_path}")
        pointer = load_json(pointer_path)
        manifest_path = (pointer_path.parent / pointer["manifest_relative"]).resolve()
    else:
        manifest_path = Path(input_manifest)
        if not manifest_path.is_absolute():
            manifest_path = project / manifest_path
        manifest_path = manifest_path.resolve()
    require(manifest_path.is_file(), f"Input manifest not found: {manifest_path}")
    manifest_sha256 = file_sha256(manifest_path)
    if pointer is not None:
        require(manifest_sha256 == pointer["manifest_sha256"], "Input manifest hash mismatch")
    manifest = load_json(manifest_path)
    require(manifest["status"] == "completed" and manifest["checks_passed"] > 0, "Input preparation is incomplete")
    require(manifest_path.parent.name == manifest["run_id"], "Input run ID mismatch")
    if pointer is not None:
        require(pointer["run_id"] == manifest["run_id"], "Input pointer run ID mismatch")
    require(manifest.get("model_version") == "fixed10m_min_EDGE_ID_common_distance_v1", "Expected fixed-10-m inputs with identical opposite-direction distances")
    require(manifest["policy"].get("opposite_distance_and_costs_identical") is True and
            manifest["policy"].get("parent_child_distance_and_costs_exactly_conserve") is True,
            "Missing input distance, cost, or parent-child conservation checks")
    folder = manifest_path.parent
    files = {x["file"]: x["sha256"] for x in manifest["output_files"]}
    for name in ("routing_arrays.npz", "arcs.csv.gz", "nodes.csv", "unique_od_tasks.csv", "od_samples.csv"):
        require(name in files and file_sha256(folder / name) == files[name], f"Input file hash mismatch: {name}")
    arcs = pd.read_csv(folder / "arcs.csv.gz", dtype=str, keep_default_na=False)
    nodes = pd.read_csv(folder / "nodes.csv", dtype=str, keep_default_na=False)
    tasks = pd.read_csv(folder / "unique_od_tasks.csv", dtype=str, keep_default_na=False)
    samples = pd.read_csv(folder / "od_samples.csv", dtype=str, keep_default_na=False)
    for frame in (tasks, samples):
        for column in ("od_task_index", "origin_node_index", "destination_node_index"):
            frame[column] = frame[column].map(int)
    tasks["sample_multiplicity"] = tasks["sample_multiplicity"].map(int)
    with np.load(folder / "routing_arrays.npz", allow_pickle=False) as data:
        net = RoutingNetwork(len(nodes), data["u"].tolist(), data["v"].tolist(),
            [int(x) for x in data["distance_exact_integer"]], arcs.physical_segment_id.tolist(),
            arcs.arc_id.tolist(), data["out_ptr"].tolist(), data["out_order"].tolist(),
            data["in_ptr"].tolist(), data["in_order"].tolist(), int(data["distance_decimal_places"]))
        hours = list(map(int, data["hours"]))
        exposures = {hour: [int(x) for x in data["exposure_exact_integer"][i]] for i, hour in enumerate(hours)}
        places = dict(zip(hours, map(int, data["exposure_decimal_places"])))
    m = len(net.u)
    require(list(arcs.arc_index.map(int)) == list(range(m)), "Arc index order mismatch")
    require(net.arc_ids == sorted(net.arc_ids) and len(set(net.arc_ids)) == m, "Arc IDs must be unique and sorted")
    require(arcs.u_index.map(int).tolist() == net.u and arcs.v_index.map(int).tolist() == net.v, "Arc endpoint mismatch")
    require(arcs.distance_exact_integer.map(int).tolist() == net.distance, "Integer distance mismatch")
    require(all(d > 0 for d in net.distance), "Arc distances must be strictly positive")
    require(all(len(costs) == m and all(c >= 0 for c in costs) for costs in exposures.values()), "Invalid exposure costs")
    require(list(nodes.node_index.map(int)) == list(range(net.n)), "Node index mismatch")
    require(tasks.od_task_index.is_unique and samples.sample_id.is_unique, "Duplicate sample or task IDs")
    require(len(samples) == manifest["counts"]["samples"], "Input sample count mismatch")
    require(len(tasks) == manifest["unique_od_tasks"], "Input task count mismatch")
    require(int(tasks.sample_multiplicity.sum()) == len(samples), "Task multiplicities do not preserve sample count")
    counts = samples.groupby("od_task_index").size()
    require(set(counts.index) == set(tasks.od_task_index)
            and counts.reindex(tasks.od_task_index).tolist() == tasks.sample_multiplicity.tolist(),
            "Sample-to-task multiplicities are inconsistent")
    node_lookup = nodes.node_id.tolist()
    for row in tasks.itertuples(index=False):
        require(node_lookup[row.origin_node_index] == row.origin_route_node and
                node_lookup[row.destination_node_index] == row.destination_route_node, "OD node mapping mismatch")
    require(hours == list(map(int, manifest["hours"])), "Input hour list mismatch")
    require(net.n == manifest["counts"]["nodes"] and m == manifest["counts"]["arcs"], "Input network size mismatch")
    return dict(net=net, hours=hours, exposure=exposures, exposure_places=places, tasks=tasks,
                samples=samples, manifest_path=str(manifest_path), manifest_sha256=manifest_sha256)


def profile_distances(inputs, path):
    net, tasks = inputs["net"], inputs["tasks"]
    if path.is_file():
        profile = pd.read_csv(path, dtype={"shortest_distance_integer": str})
        require(profile.od_task_index.tolist() == tasks.od_task_index.tolist(), "Cached distance profile differs from tasks")
        require(len(profile) == len(tasks), "Cached distance profile is incomplete")
        return profile
    # This scalar graph is ONLY for selecting OD distance quantiles.
    # The exact multiobjective solver above always uses every original directed arc.
    graph = nx.DiGraph()
    graph.add_nodes_from(range(net.n))
    for edge, (u, v) in enumerate(zip(net.u, net.v)):
        if not graph.has_edge(u, v) or net.distance[edge] < graph[u][v]["weight"]:
            graph.add_edge(u, v, weight=net.distance[edge])
    rows = []
    for i, task in enumerate(tasks.itertuples(index=False), 1):
        distance, _ = nx.bidirectional_dijkstra(graph, task.origin_node_index, task.destination_node_index)
        rows.append(dict(od_task_index=task.od_task_index, shortest_distance_integer=str(distance),
                         shortest_distance_m=distance / 10**net.distance_places))
        if i % 100 == 0 or i == len(tasks):
            print(f"Distance profile: {i:,}/{len(tasks):,}", flush=True)
    del graph
    gc.collect()
    atomic_csv(path, rows)
    return pd.DataFrame(rows)


def task_summary(result, net, c_places):
    row = {key: result[key] for key in ("od_task_index", "hour", "status")}
    row.update(result.get("stats", {}))
    row["complete_frontier_within_20pct"] = result["complete_frontier_within_20pct"]
    row["reason"] = result.get("reason", "")
    if result["status"] == "completed":
        shortest, minimum = result["shortest"], result["minimum_exposure_unbounded"]
        row.update(shortest_distance_m=int(shortest["distance_integer"]) / 10**net.distance_places,
            shortest_exposure=int(shortest["exposure_integer"]) / 10**c_places,
            minimum_exposure_unbounded=int(minimum["exposure_integer"]) / 10**c_places,
            minimum_exposure_route_distance_m=int(minimum["distance_integer"]) / 10**net.distance_places,
            multiple_routes=result["multiple_routes"], pec_segments=len(result["pec_physical_segments"]))
    return row


def export_tables(run, inputs, keys, solver_fingerprint):
    summaries, frontiers, budgets, memberships, files = [], [], [], [], []
    net = inputs["net"]
    for task_index, hour in keys:
        path = run / "tasks" / f"od_{task_index:05d}_hour_{hour}.json.gz"
        if not path.is_file():
            summaries.append(dict(od_task_index=task_index, hour=hour, status="not_run",
                                  complete_frontier_within_20pct=False))
            continue
        result = load_json(path)
        require(result["input_manifest_sha256"] == inputs["manifest_sha256"] and
                result["solver_fingerprint"] == solver_fingerprint, "Task output provenance mismatch")
        files.append(dict(file=str(path.relative_to(run)), sha256=file_sha256(path)))
        c_places = inputs["exposure_places"][hour]
        summaries.append(task_summary(result, net, c_places))
        if result["status"] != "completed":
            continue
        dag = result["dag"]
        for point_index, terminal in enumerate(dag["terminals"]):
            d, c = dag["distance_integer"][terminal], dag["exposure_integer"][terminal]
            frontiers.append(dict(od_task_index=task_index, hour=hour, point_index=point_index,
                terminal_label=terminal, distance_integer=d, exposure_integer=c,
                distance_m=int(d) / 10**net.distance_places, exposure=int(c) / 10**c_places))
        for pct, route in result["selected_budget_routes"].items():
            budgets.append(dict(od_task_index=task_index, hour=hour, detour_percent=int(pct),
                distance_integer=route["distance_integer"], exposure_integer=route["exposure_integer"],
                distance_m=int(route["distance_integer"]) / 10**net.distance_places,
                exposure=int(route["exposure_integer"]) / 10**c_places,
                terminal_label=route["terminal"]))
        memberships.extend(dict(od_task_index=task_index, hour=hour, physical_segment_id=x)
                           for x in result["pec_physical_segments"])
    atomic_csv(run / "task_summary.csv", summaries)
    atomic_csv(run / "frontier_points.csv", frontiers,
        ["od_task_index", "hour", "point_index", "terminal_label", "distance_integer", "exposure_integer", "distance_m", "exposure"])
    atomic_csv(run / "budget_optima.csv", budgets,
        ["od_task_index", "hour", "detour_percent", "distance_integer", "exposure_integer", "distance_m", "exposure", "terminal_label"])
    atomic_csv(run / "pec_membership.csv", memberships, ["od_task_index", "hour", "physical_segment_id"])
    summary = pd.DataFrame(summaries)
    sample_results = inputs["samples"].merge(summary, on="od_task_index", how="inner", validate="many_to_many")
    require(not sample_results.duplicated(["sample_id", "hour"]).any(), "Duplicate sample-hour records")
    require(len(sample_results) == sum(int(inputs["tasks"].set_index("od_task_index").loc[t, "sample_multiplicity"])
                                      for t, h in keys), "Expanded sample count mismatch")
    atomic_csv(run / "sample_hour_results.csv", sample_results)
    atomic_json(run / "task_file_index.json", files)
    export_diagnostics(run, inputs, summaries, frontiers)
    return summaries, len(sample_results)


In [8]:
def environment_info():
    cpu_model = platform.processor().strip()
    if sys.platform == "win32":
        try:
            import winreg
            with winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE,
                                r"HARDWARE\DESCRIPTION\System\CentralProcessor\0") as key:
                cpu_model = str(winreg.QueryValueEx(key, "ProcessorNameString")[0]).strip()
        except OSError:
            pass
    elif sys.platform.startswith("linux"):
        try:
            for line in Path("/proc/cpuinfo").read_text().splitlines():
                if line.lower().startswith("model name"):
                    cpu_model = line.split(":", 1)[1].strip()
                    break
        except OSError:
            pass
    try:
        affinity = psutil.Process().cpu_affinity()
    except (AttributeError, OSError, psutil.Error):
        affinity = None
    return dict(
        python=sys.version, platform=platform.platform(), machine=platform.machine(),
        cpu_model=cpu_model or None, physical_cores=psutil.cpu_count(logical=False),
        logical_processors=psutil.cpu_count(logical=True), cpu_affinity=affinity,
        total_memory_gib=psutil.virtual_memory().total / 1024**3,
        routing_workers=1, numpy=np.__version__, pandas=pd.__version__,
        scipy=scipy.__version__, networkx=nx.__version__, psutil=psutil.__version__,
        thread_environment={name: os.environ.get(name) for name in
                            ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")})


def export_diagnostics(run, inputs, summaries, frontiers):
    """Front sizes count objective pairs; runtime statistics count solved tasks."""
    multiplicity = inputs["tasks"].set_index("od_task_index").sample_multiplicity.to_dict()
    points = {}
    for row in frontiers:
        key = (int(row["od_task_index"]), int(row["hour"]))
        points.setdefault(key, []).append(int(row["distance_integer"]))
    sizes = []
    for (task_index, hour), distances in points.items():
        shortest_distance = min(distances)
        for pct in BUDGET_PERCENTAGES:
            bound = (100 + pct) * shortest_distance // 100
            sizes.append(dict(od_task_index=task_index, hour=hour, detour_percent=pct,
                              frontier_points=sum(d <= bound for d in distances),
                              sample_multiplicity=int(multiplicity[task_index])))
    columns = ["od_task_index", "hour", "detour_percent", "frontier_points", "sample_multiplicity"]
    atomic_csv(run / "frontier_sizes.csv", sizes, columns)
    size_frame = pd.DataFrame(sizes, columns=columns)
    aggregates = []
    for pct in BUDGET_PERCENTAGES:
        for hour in ["all", *inputs["hours"]]:
            group = size_frame.loc[size_frame.detour_percent.eq(pct)]
            if hour != "all":
                group = group.loc[group.hour.eq(hour)]
            for population in ("unique_od_hour_tasks", "sample_hour_observations"):
                values = group.frontier_points.to_numpy(dtype=int)
                if population == "sample_hour_observations":
                    values = np.repeat(values, group.sample_multiplicity.to_numpy(dtype=int))
                row = dict(hour=hour, detour_percent=pct, population=population,
                           completed_observations=len(values))
                if len(values):
                    row.update(mean_frontier_points=float(np.mean(values)),
                               median_frontier_points=float(np.median(values)),
                               p90_frontier_points=float(np.percentile(values, 90)),
                               maximum_frontier_points=int(np.max(values)),
                               one_point_pct=float(100 * np.mean(values == 1)),
                               two_points_pct=float(100 * np.mean(values == 2)),
                               at_least_three_points_pct=float(100 * np.mean(values >= 3)))
                aggregates.append(row)
    atomic_csv(run / "frontier_size_summary.csv", aggregates)

    runtimes = []
    for hour in ["all", *inputs["hours"]]:
        group = [r for r in summaries if hour == "all" or r["hour"] == hour]
        valid = [r for r in group if r["status"] == "completed"]
        row = dict(hour=hour, requested_tasks=len(group), completed_tasks=len(valid),
                   incomplete_or_not_run=len(group) - len(valid))
        if valid:
            values = np.asarray([r["seconds"] for r in valid], dtype=float)
            row.update(mean_task_seconds=float(np.mean(values)),
                       median_task_seconds=float(np.median(values)),
                       p95_task_seconds=float(np.percentile(values, 95)),
                       maximum_task_seconds=float(np.max(values)),
                       sum_completed_task_seconds=float(np.sum(values)),
                       maximum_labels_created=max(r["labels_created"] for r in valid),
                       maximum_sampled_process_rss_gib=max(
                           r["sampled_peak_process_rss_gib"] for r in valid))
        runtimes.append(row)
    atomic_csv(run / "runtime_summary.csv", runtimes)


def save_session(run, session, session_started):
    session["wall_clock_seconds"] = time.perf_counter() - session_started
    session["finished_utc"] = datetime.now(timezone.utc).isoformat()
    atomic_json(run / "sessions" / (session["session_id"] + ".json"), session)
    sessions = [load_json(path) for path in sorted((run / "sessions").glob("*.json"))]
    complete_times = all(s.get("finished_utc") and s.get("wall_clock_seconds") is not None
                         for s in sessions)
    fresh_complete = (len(sessions) == 1 and session["status"] == "completed_full"
                      and session["reused_tasks"] == 0)
    return dict(
        session_count=len(sessions), current_session_wall_seconds=session["wall_clock_seconds"],
        cumulative_recorded_session_wall_seconds=sum(s.get("wall_clock_seconds", 0) for s in sessions),
        all_session_times_recorded=complete_times,
        fresh_full_run_wall_seconds=session["wall_clock_seconds"] if fresh_complete else None,
        wall_clock_scope="Input checks, validation, routing, checkpoint writes and table export; excludes idle time between sessions.",
        task_seconds_scope="Two reverse shortest-path searches, label search, DAG construction, shared segments and budget routes; excludes file I/O.")


In [9]:
def source_fingerprint():
    import ast
    import inspect
    import textwrap

    parts = [CODE_FINGERPRINT, SOLVER_VERSION, str(BUDGET_PERCENTAGES)]
    for name in FINGERPRINT_FUNCTIONS:
        function = globals()[name]
        try:
            source = textwrap.dedent(inspect.getsource(function))
            parts.append(ast.dump(ast.parse(source), include_attributes=False))
        except (OSError, TypeError):
            pass
    return hashlib.sha256("\n".join(parts).encode("utf-8")).hexdigest()


def run_routing(project, input_manifest, mode="full", pilot_od_count=20,
                limits=None, resume_run=None, progress_every=100, main_detour_percent=20,
                output_root=None):
    project = Path(project).resolve()
    require(project.is_dir(), f"Project directory not found: {project}")
    require(mode in ("pilot", "full"), "MODE must be 'pilot' or 'full'.")
    require(isinstance(pilot_od_count, int) and pilot_od_count > 0, "PILOT_OD_COUNT must be positive.")
    require(isinstance(progress_every, int) and progress_every > 0, "PROGRESS_EVERY must be positive.")
    require(main_detour_percent in BUDGET_PERCENTAGES, "Main detour cap is not in BUDGET_PERCENTAGES.")
    limits = dict(limits if limits is not None else LIMITS)
    require(set(limits) == {"seconds", "labels", "predecessor_links", "rss_gib", "available_gib"},
            "Unexpected resource-limit settings.")
    require(all(limits[k] > 0 for k in limits if k != "available_gib")
            and limits["available_gib"] >= 0, "Invalid resource limits.")
    output_root = Path(output_root) if output_root is not None else project / "L_revision"
    if not output_root.is_absolute():
        output_root = project / output_root
    stage = output_root.resolve() / "exact_pareto_routing"
    session_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    if resume_run is None:
        run = stage / "runs" / session_id
        run.mkdir(parents=True, exist_ok=False)
    else:
        run = Path(resume_run).resolve()
        require((run / "manifest.json").is_file(), f"Run manifest not found: {run}")
        require(run.parent == (stage / "runs").resolve(), "RESUME_RUN is outside the routing runs directory.")
    fingerprint = source_fingerprint()
    session_started = time.perf_counter()
    environment = environment_info()
    session = dict(session_id=session_id, status="initializing", limits=limits,
                   started_utc=datetime.now(timezone.utc).isoformat(),
                   solved_tasks=0, reused_tasks=0, solve_seconds=0.0, environment=environment)
    status = dict(status="initializing", run_id=run.name, output_directory=str(run),
                  output_root=str(stage.parent),
                  mode=mode, pilot_od_count=pilot_od_count, limits=limits,
                  solver_version=SOLVER_VERSION, solver_fingerprint=fingerprint,
                  source_code_sha256=CODE_FINGERPRINT, started_utc=session["started_utc"],
                  frontier_detour_percent=20, main_detour_cap_percent=main_detour_percent,
                  budget_percentages=list(BUDGET_PERCENTAGES),
                  minimum_exposure_baseline_has_distance_cap=False, environment=environment)
    identity_verified = False
    print(f"Run: {run}", flush=True)
    try:
        previous = load_json(run / "manifest.json") if resume_run is not None else None
        inputs = load07(project, input_manifest)
        status.update(input_manifest=inputs["manifest_path"], input_manifest_sha256=inputs["manifest_sha256"])
        if previous is not None:
            for key in ("mode", "pilot_od_count", "solver_fingerprint", "input_manifest_sha256",
                        "main_detour_cap_percent"):
                require(previous.get(key) == status[key], f"Resume mismatch: {key}. Start a new run.")
            status["original_started_utc"] = previous.get("original_started_utc", previous["started_utc"])
        identity_verified = True
        atomic_json(run / "manifest.json", status)
        atomic_json(run / "environment.json", environment)
        atomic_json(run / "input_manifest_snapshot.json", load_json(inputs["manifest_path"]))
        atomic_json(run / "sessions" / (session_id + ".json"), session)
        validation = run_algorithm_validation()
        atomic_json(run / "algorithm_validation.json", validation)
        print(f"Validation: {validation['fixture_count']} fixtures, {validation['milp_budget_checks']} MILP checks.", flush=True)

        tasks, net = inputs["tasks"], inputs["net"]
        if mode == "pilot":
            profile = profile_distances(inputs, run / "od_distance_profile.csv")
            ranked = sorted(zip(profile.shortest_distance_integer.map(int), profile.od_task_index.map(int)))
            positions = np.linspace(0, len(ranked) - 1, min(pilot_od_count, len(ranked)), dtype=int)
            selected = tasks.set_index("od_task_index").loc[[ranked[p][1] for p in positions]].reset_index()
            selected = selected.merge(profile, on="od_task_index", validate="one_to_one", sort=False)
        else:
            selected = tasks.copy()
        require(len(selected) > 0, "No OD tasks selected.")
        atomic_csv(run / "selected_od_tasks.csv", selected)
        keys = [(int(t), hour) for t in selected.od_task_index for hour in inputs["hours"]]
        status.update(status="running", selected_od_count=len(selected), requested_tasks=len(keys))
        session["status"] = "running"
        atomic_json(run / "manifest.json", status)
        atomic_json(run / "sessions" / (session_id + ".json"), session)
        print(f"{mode}: {len(selected):,} OD pairs, {len(inputs['hours'])} hours, {len(keys):,} tasks.", flush=True)
        (run / "tasks").mkdir(exist_ok=True)
        summaries, completed = [], 0
        task_lookup = tasks.set_index("od_task_index")
        for number, (task_index, hour) in enumerate(keys, 1):
            path = run / "tasks" / f"od_{task_index:05d}_hour_{hour}.json.gz"
            result = None
            row = task_lookup.loc[task_index]
            if path.is_file():
                existing = load_json(path)
                require(existing["input_manifest_sha256"] == inputs["manifest_sha256"]
                        and existing["solver_fingerprint"] == fingerprint,
                        f"Task provenance mismatch: {path.name}")
                require(existing["od_task_index"] == task_index and existing["hour"] == hour
                        and existing["origin_node_index"] == int(row.origin_node_index)
                        and existing["destination_node_index"] == int(row.destination_node_index),
                        f"Task identity mismatch: {path.name}")
                if existing["status"] == "completed":
                    result = existing
                    session["reused_tasks"] += 1
            if result is None:
                result = exact_pareto(net, inputs["exposure"][hour],
                                      int(row.origin_node_index), int(row.destination_node_index), limits)
                session["solved_tasks"] += 1
                session["solve_seconds"] += result["stats"]["seconds"]
                result.update(od_task_index=task_index, hour=hour, input_manifest_sha256=inputs["manifest_sha256"],
                              solver_version=SOLVER_VERSION, solver_fingerprint=fingerprint,
                              origin_node_index=int(row.origin_node_index), destination_node_index=int(row.destination_node_index),
                              distance_decimal_places=net.distance_places, exposure_decimal_places=inputs["exposure_places"][hour],
                              limits=limits, session_id=session_id, finished_utc=datetime.now(timezone.utc).isoformat())
                atomic_json(path, result)
            summary = task_summary(result, net, inputs["exposure_places"][hour])
            summaries.append(summary)
            good = result["status"] == "completed"
            completed += int(good)
            if number % progress_every == 0 or number == len(keys) or not good:
                atomic_csv(run / "task_summary.csv", summaries)
                elapsed = time.perf_counter() - session_started
                print(f"Tasks {number:,}/{len(keys):,}; completed {completed:,}; elapsed {elapsed:.1f} s.", flush=True)
            if not good:
                print(f"Incomplete: OD {task_index}, hour {hour}: {result.get('reason', '')}", flush=True)
            del result
            gc.collect()
            if not good and summary.get("last_phase") == "initializing":
                break

        summaries, sample_rows = export_tables(run, inputs, keys, fingerprint)
        completed = sum(r["status"] == "completed" for r in summaries)
        complete = completed == len(keys)
        full_complete = mode == "full" and complete
        final_status = ("completed_full" if full_complete else "completed_pilot") if complete else "incomplete"
        status.update(status=final_status, completed_tasks=completed, incomplete_or_not_run=len(keys) - completed,
                      sample_hour_rows=sample_rows, all_2500_samples_all_hours_complete=full_complete,
                      finished_utc=datetime.now(timezone.utc).isoformat())
        valid = [r for r in summaries if r["status"] == "completed"]
        if valid:
            seconds = [r["seconds"] for r in valid]
            status["timing_summary"] = dict(median_task_seconds=float(np.median(seconds)),
                p95_task_seconds=float(np.percentile(seconds, 95)), maximum_task_seconds=max(seconds),
                sum_completed_task_seconds=float(sum(seconds)),
                maximum_labels_created=max(r["labels_created"] for r in valid),
                maximum_sampled_process_rss_gib=max(r["sampled_peak_process_rss_gib"] for r in valid))
        session.update(status=final_status, completed_tasks=completed)
        status["runtime"] = save_session(run, session, session_started)
        atomic_json(run / "manifest.json", status)
        if complete:
            pointer_name = "LATEST_FULL_SUCCESS.json" if full_complete else "LATEST_PILOT_SUCCESS.json"
            atomic_json(stage / pointer_name, dict(run_id=run.name, manifest=str(run / "manifest.json"),
                                                  manifest_sha256=file_sha256(run / "manifest.json")))
        print(f"{final_status}: {completed:,}/{len(keys):,} tasks; {session['wall_clock_seconds']:.1f} s this session.", flush=True)
        return run
    except (Exception, KeyboardInterrupt) as error:
        status.update(status="interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
                      error_type=type(error).__name__, error=str(error),
                      finished_utc=datetime.now(timezone.utc).isoformat())
        if resume_run is None or identity_verified:
            session.update(status=status["status"], error_type=type(error).__name__, error=str(error))
            status["runtime"] = save_session(run, session, session_started)
            atomic_json(run / "manifest.json", status)
            atomic_json(run / "last_error.json", dict(session_id=session_id, status=status["status"],
                        error_type=type(error).__name__, error=str(error), traceback=traceback.format_exc()))
        print(f"{status['status']}: {error}", flush=True)
        raise


In [10]:
CODE_FINGERPRINT = "c3127981648bd4ec8e075e088223cc04923fbfdba780dd642bf47af50243815c"
FINGERPRINT_FUNCTIONS = ('require', 'file_sha256', 'atomic_json', 'load_json', 'atomic_csv', 'RoutingNetwork', 'ResourceLimit', 'Monitor', 'reverse_lex_dijkstra', 'restore_baseline', 'check_route', 'compact_dag', 'representative_route', 'physical_intersection', 'exact_pareto', 'tiny_network', 'enumerate_simple_routes', 'enumerate_tiny_dag', 'milp_budget_value', 'run_algorithm_validation', 'load07', 'profile_distances', 'task_summary', 'export_tables', 'environment_info', 'export_diagnostics', 'save_session', 'source_fingerprint', 'run_routing')


In [11]:
OUTPUT_DIR = run_routing(
    PROJECT,
    input_manifest=INPUT_MANIFEST,
    mode=MODE,
    pilot_od_count=PILOT_OD_COUNT,
    limits=LIMITS,
    resume_run=RESUME_RUN,
    progress_every=PROGRESS_EVERY,
    main_detour_percent=MAIN_DETOUR_PERCENT,
    output_root=OUTPUT_ROOT,
)


Run: C:\Users\LX\Desktop\quebec\L_revision\exact_pareto_routing\runs\20261021T194025_318746Z
Validation: 29 fixtures, 108 MILP checks.
full: 2,492 OD pairs, 5 hours, 12,460 tasks.
Tasks 100/12,460; completed 100; elapsed 152.1 s.
Tasks 200/12,460; completed 200; elapsed 232.7 s.
Tasks 300/12,460; completed 300; elapsed 313.1 s.
Tasks 400/12,460; completed 400; elapsed 392.5 s.
Tasks 500/12,460; completed 500; elapsed 473.6 s.
Tasks 600/12,460; completed 600; elapsed 552.8 s.
Tasks 700/12,460; completed 700; elapsed 642.6 s.
Tasks 800/12,460; completed 800; elapsed 725.5 s.
Tasks 900/12,460; completed 900; elapsed 806.3 s.
Tasks 1,000/12,460; completed 1,000; elapsed 886.9 s.
Tasks 1,100/12,460; completed 1,100; elapsed 967.4 s.
Tasks 1,200/12,460; completed 1,200; elapsed 1056.0 s.
Tasks 1,300/12,460; completed 1,300; elapsed 1136.8 s.
Tasks 1,400/12,460; completed 1,400; elapsed 1219.8 s.
Tasks 1,500/12,460; completed 1,500; elapsed 1302.7 s.
Tasks 1,600/12,460; completed 1,600; elaps